In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_rel
import warnings
warnings.filterwarnings("ignore")
from scipy.stats import skew
from scipy.stats import kurtosis
from scipy.stats import wilcoxon

# Test de hipótesis para sulfuros primarios de cobre segmentado por control de calidad

## 1) Test de hipótesis para extracción de cobre mediante tierras

### 1.1) Test de hipotesis normalidad para sulfuros secundarios de cobre metodo de extraccion por tierras

In [6]:
import pandas as pd
from scipy import stats

# 1. Definición de columnas y carga de datos
columnas_deseadas = ['Zona Min', 'ext_cut_rip_ph', 'ext_cut_rip_selec_col']
excelPath = r"C:\Users\Diego\Desktop\Ultimo Esfuerzo\Copia de Base de Datos Cuprochlor Sin T Final.xlsx"

df3 = pd.read_excel(excelPath, "BD elim y reducido (3)", usecols=columnas_deseadas)

# 2. Cálculo de la variable diferencia (Iso pH - Columna)
df3['diferencia'] = df3['ext_cut_rip_ph'] - df3['ext_cut_rip_selec_col']

# 3. Filtrado por zona mineralógica (Sulfuros Primarios - PRI)
# Limpiamos valores nulos específicos de este segmento para la variable 'diferencia'
datos_pri = df3[df3['Zona Min'] == 'PRI']['diferencia'].dropna()

# 3.Calculo de asimetría 
datos=datos_pri
asimetria=skew(datos)
print("Asimetria:",asimetria)


# 4.Calculo de Curtosis
datos=datos_pri
curtosis=kurtosis(datos)
print("Curtosis:",curtosis)

# 4. Realizar el Test de Shapiro-Wilk
# Ideal para muestras pequeñas (n < 50)
shapiro_stat, p_value = stats.shapiro(datos_pri)

# 5. Mostrar e interpretar resultados
print("--- TEST DE NORMALIDAD (SHAPIRO-WILK) - ZONA: PRI (SULFUROS PRIMARIOS) ---")
print(f"Estadístico W: {shapiro_stat:.4f}")
print(f"P-valor: {p_value:.4f}")
print(f"Cantidad de muestras analizadas en PRI: {len(datos_pri)}")

print("\nInterpretación:")
alpha = 0.05
if p_value > alpha:
    print("Resultado: Los datos de Sulfuros Primarios (PRI) parecen seguir una distribución NORMAL.")
    print("Interpretación: No se rechaza la hipótesis nula (p > 0.05).")
else:
    print("Resultado: Los datos de Sulfuros Primarios (PRI) NO siguen una distribución normal.")
    print("Interpretación: Se7 rechaza la hipótesis nula (p < 0.05).")

Asimetria: 0.08924361629846948
Curtosis: -1.099102498238248
--- TEST DE NORMALIDAD (SHAPIRO-WILK) - ZONA: PRI (SULFUROS PRIMARIOS) ---
Estadístico W: 0.9701
P-valor: 0.8988
Cantidad de muestras analizadas en PRI: 7

Interpretación:
Resultado: Los datos de Sulfuros Primarios (PRI) parecen seguir una distribución NORMAL.
Interpretación: No se rechaza la hipótesis nula (p > 0.05).


### 1.2) Test t Student para sulfuros primarios de cobre metodo de extraccion por tierras

In [10]:
import pandas as pd
from scipy import stats

# 1. Definición de columnas y carga de datos
columnas_deseadas = ['Zona Min', 'ext_cut_rip_ph', 'ext_cut_rip_selec_col']
excelPath = r"C:\Users\Diego\Desktop\Ultimo Esfuerzo\Copia de Base de Datos Cuprochlor Sin T Final.xlsx"

df3 = pd.read_excel(excelPath, "BD elim y reducido (3)", usecols=columnas_deseadas)

# 2. Filtrado por zona (PRI - Sulfuros Primarios) y limpieza de nulos
# Filtramos la zona y aseguramos que existan ambos datos para la comparación por pares
df_pri = df3[df3['Zona Min'] == 'PRI'].dropna(subset=['ext_cut_rip_ph', 'ext_cut_rip_selec_col'])

# 3. Realizar el Test T de Student para muestras relacionadas (Paired T-Test)
t_stat, p_value = stats.ttest_rel(df_pri['ext_cut_rip_ph'], df_pri['ext_cut_rip_selec_col'])

# 4. Cálculo de medias reales para la zona PRI
media_ph = df_pri['ext_cut_rip_ph'].mean()
media_col = df_pri['ext_cut_rip_selec_col'].mean()

# 5. Mostrar e interpretar resultados con formato de redacción específico
print("--- TEST T DE STUDENT (MUESTRAS RELACIONADAS) - ZONA: PRI (SULFUROS PRIMARIOS) ---")
print(f"Estadístico t: {t_stat:.4f}")
print(f"P-valor: {p_value:.10f}")
print(f"Cantidad de muestras analizadas en PRI: {len(df_pri)}")

print("\nInterpretación:")
alpha = 0.05
if p_value < alpha:
    print("Resultado: SIGNIFICATIVO.")
    
    # Redacción dinámica basada en los valores reales de la zona
    if media_ph > media_col:
        print(f"La extracción Iso pH ({media_ph:.2f}%) es significativamente superior a la de Columna ({media_col:.2f}%).")
    else:
        print(f"La extracción por Columna ({media_col:.2f}%) es significativamente superior a la de Iso pH ({media_ph:.2f}%).")
else:
    print("Resultado: NO SIGNIFICATIVO.")
    print(f"No hay evidencia suficiente para afirmar una diferencia. (Medias: Iso pH {media_ph:.2f}% vs Columna {media_col:.2f}%)")

--- TEST T DE STUDENT (MUESTRAS RELACIONADAS) - ZONA: PRI (SULFUROS PRIMARIOS) ---
Estadístico t: 1.1340
P-valor: 0.3000693581
Cantidad de muestras analizadas en PRI: 7

Interpretación:
Resultado: NO SIGNIFICATIVO.
No hay evidencia suficiente para afirmar una diferencia. (Medias: Iso pH 42.01% vs Columna 36.74%)


## 2) Test de hipótesis para extracción de cobre mediante cabeza analizada

### 2.1) Test de hipotesis normalidad para sulfuros primarios de cobre metodo cabeza analizada

In [13]:
import pandas as pd
from scipy import stats
from scipy.stats import skew, kurtosis

# 1. Definición de columnas y carga de datos (Análisis de Solución)
columnas_deseadas = ['Zona Min', 'ext_cut_anlz_ph', 'ext_cut_anlz_col']
excelPath = r"C:\Users\Diego\Desktop\Ultimo Esfuerzo\Copia de Base de Datos Cuprochlor Sin T Final.xlsx"

# Cargamos el DataFrame df4
df4 = pd.read_excel(excelPath, "BD elim y reducido (3)", usecols=columnas_deseadas)

# 2. Cálculo de la variable diferencia (Iso pH - Columna en Solución)
df4['diferencia'] = df4['ext_cut_anlz_ph'] - df4['ext_cut_anlz_col']

# 3. Filtrado por zona mineralógica (Sulfuros Primarios - PRI)
# Limpiamos valores nulos específicos de este segmento
datos_pri = df4[df4['Zona Min'] == 'PRI']['diferencia'].dropna()

# 4. Cálculo de Asimetría
asimetria = skew(datos_pri)
print(f"Asimetría (PRI - Solución): {asimetria:.4f}")

# 5. Cálculo de Curtosis
curto = kurtosis(datos_pri)
print(f"Curtosis (PRI - Solución): {curto:.4f}")

# 6. Realizar el Test de Shapiro-Wilk
# Recomendado para PRI debido al tamaño de muestra (n < 50)
shapiro_stat, p_value = stats.shapiro(datos_pri)

# 7. Mostrar e interpretar resultados
print("\n--- TEST DE NORMALIDAD (SHAPIRO-WILK) - ZONA: PRI (SULFUROS PRIMARIOS) ---")
print(f"Estadístico W: {shapiro_stat:.4f}")
print(f"P-valor: {p_value:.4f}")
print(f"Cantidad de muestras analizadas en PRI: {len(datos_pri)}")

print("\nInterpretación:")
alpha = 0.05
if p_value > alpha:
    print("Resultado: Los datos de Sulfuros Primarios (PRI) parecen seguir una distribución NORMAL.")
    print("Interpretación: No se rechaza la hipótesis nula (p > 0.05).")
else:
    print("Resultado: Los datos de Sulfuros Primarios (PRI) NO siguen una distribución normal.")
    print("Interpretación: Se rechaza la hipótesis nula (p < 0.05).")

Asimetría (PRI - Solución): -1.3655
Curtosis (PRI - Solución): 0.4877

--- TEST DE NORMALIDAD (SHAPIRO-WILK) - ZONA: PRI (SULFUROS PRIMARIOS) ---
Estadístico W: 0.7886
P-valor: 0.0315
Cantidad de muestras analizadas en PRI: 7

Interpretación:
Resultado: Los datos de Sulfuros Primarios (PRI) NO siguen una distribución normal.
Interpretación: Se rechaza la hipótesis nula (p < 0.05).


### 2.2) Test t Student para la extracción de sulfuros primarios de cobre calculada mediante método de cabeza analizada

In [17]:
import pandas as pd
from scipy import stats

# 1. Definición de columnas y carga de datos (Análisis de Solución)
columnas_deseadas = ['Zona Min', 'ext_cut_anlz_ph', 'ext_cut_anlz_col']
excelPath = r"C:\Users\Diego\Desktop\Ultimo Esfuerzo\Copia de Base de Datos Cuprochlor Sin T Final.xlsx"

# Cargamos el DataFrame df4
df4 = pd.read_excel(excelPath, "BD elim y reducido (2)", usecols=columnas_deseadas)

# 2. Filtrado por zona (PRI - Sulfuros Primarios) y limpieza de nulos
# Aseguramos que existan ambos datos de solución para la comparación por pares
df_pri_anlz = df4[df4['Zona Min'] == 'PRI'].dropna(subset=['ext_cut_anlz_ph', 'ext_cut_anlz_col'])

# 3. Realizar el Test T de Student para muestras relacionadas (Paired T-Test)
t_stat, p_value = stats.ttest_rel(df_pri_anlz['ext_cut_anlz_ph'], df_pri_anlz['ext_cut_anlz_col'])

# 4. Mostrar e interpretar resultados con redacción dinámica
print("--- TEST T DE STUDENT (MUESTRAS RELACIONADAS) - ZONA: PRI (SULFUROS PRIMARIOS - SOLUCIÓN) ---")
print(f"Estadístico t: {t_stat:.4f}")
print(f"P-valor: {p_value:.10f}")
print(f"Cantidad de muestras analizadas en PRI: {len(df_pri_anlz)}")

print("\nInterpretación:")
alpha = 0.05
if p_value < alpha:
    print("Resultado: SIGNIFICATIVO.")
    print("En la zona de Sulfuros Primarios (PRI), existe una diferencia estadística real en la extracción medida vía solución.")
    
    # Cálculo de medias para determinar eficiencia
    media_ph = df_pri_anlz['ext_cut_anlz_ph'].mean()
    media_col = df_pri_anlz['ext_cut_anlz_col'].mean()
    
    if media_ph > media_col:
        print(f"La extracción Iso pH ({media_ph:.2f}%) es significativamente superior a la de Columna ({media_col:.2f}%).")
    else:
        print(f"La extracción por Columna ({media_col:.2f}%) es significativamente superior a la de Iso pH ({media_ph:.2f}%).")
else:
    print("Resultado: NO SIGNIFICATIVO.")
    print("En la zona PRI (solución), no hay evidencia suficiente para afirmar que un método sea diferente al otro (p > 0.05).")

--- TEST T DE STUDENT (MUESTRAS RELACIONADAS) - ZONA: PRI (SULFUROS PRIMARIOS - SOLUCIÓN) ---
Estadístico t: 1.9770
P-valor: 0.0635601044
Cantidad de muestras analizadas en PRI: 19

Interpretación:
Resultado: NO SIGNIFICATIVO.
En la zona PRI (solución), no hay evidencia suficiente para afirmar que un método sea diferente al otro (p > 0.05).
